In [1]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Authenticate
login(hf_token)

In [1]:
import os
import gc
import torch
import torch.nn.functional as F
import math
import numpy as np
import json
from glob import glob
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM

# --- FUNÇÃO AUXILIAR PARA LIMPAR VRAM ---
def limpar_memoria():
    """Força a limpeza da memória RAM e VRAM (GPU)."""
    gc.collect()                # Limpa objetos Python não utilizados
    torch.cuda.empty_cache()    # Limpa o cache de alocação do PyTorch
    print("🧹 Memória limpa!")

# --- Configuração do Dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Ambiente de Execução: {device.upper()}")

/home/matheus/Área de trabalho/monolingual_and_multilingual/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Ambiente de Execução: CUDA


In [2]:
def load_model_and_tokenizer(model_name:str, model_type:AutoModelForCausalLM | AutoModelForMaskedLM):
    if model_type is not AutoModelForCausalLM and model_type is not AutoModelForMaskedLM:
        raise Exception("Model type must be `AutoModelForCausalLM` or `AutoModelForMaskedLM` ")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = model_type.from_pretrained(
        model_name,
        #device_map="auto",
        #torch_dtype=torch.float16
    )

    model.to(device)
    model.eval()

    return tokenizer, model

In [3]:
CORPORA_FOLDER = "./Universal Dependencies/"
OUTPUT_FOLDER = "./results/"

os.makedirs(CORPORA_FOLDER, exist_ok = True)
os.makedirs(OUTPUT_FOLDER, exist_ok = True)


models = {
    # Casual LM
    "Gemma": "google/gemma-2-2b-it",
    "Qwen" : "Qwen/Qwen3-Embedding-0.6B",
    "Tucano": "TucanoBR/Tucano-630m",
    "Gervasio": "PORTULAN/gervasio-8b-portuguese-ptpt-decoder",

    # Masked LM
    "BERTimbau": "neuralmind/bert-base-portuguese-cased",
    "BERTimbauLarge": "neuralmind/bert-large-portuguese-cased",
    "Albertina100m": "PORTULAN/albertina-100m-portuguese-ptbr-encoder",
    "Albertina900m": "PORTULAN/albertina-900m-portuguese-ptbr-encoder",
    "Serafim335m": "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder"
}

# Casual LM - Perplexity

In [ ]:
def eval_causal_sliding(model, tokenizer, dataset, stride=512):
    results = {}

    max_len = model.config.max_position_embeddings

    # The perplexity is calculated for each text
    for label, texts in dataset.items():
        text_results = []

        for i, text in enumerate(texts):

            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            num_tokens_text = inputs.input_ids.size(1)

            nlls = []

            total_nll = 0.0
            total_tokens = 0
            prev_end_window = 0

            # Sliding window
            for begin_window in range(0, num_tokens_text, stride):
                end_window = min(begin_window + max_len, num_tokens_text)

                # Since the windows overlap, this indicates how many new tokens
                # will be actually used to calculate the perplexity
                target_len = end_window - prev_end_window

                # Select only the tokens from the window
                input_ids = inputs.input_ids[:, begin_window:end_window]
                target_ids = input_ids.clone()

                # Loss will ignore labels with value -100
                target_ids[:, :-target_len] = -100

                with torch.no_grad():
                    outputs = model(input_ids, labels=target_ids)
                    neg_log_likelihood = outputs.loss

                total_nll += neg_log_likelihood.item() * target_len
                total_tokens += target_len
                prev_end_window = end_window

                # nlls.append(neg_log_likelihood)
                if end_window == num_tokens_text:
                    break

            ppl = math.exp(total_nll / total_tokens)

            text_results.append({
                "text_id": i,
                "num_tokens": target_len,
                "ppl": ppl
            })

        avg_ppl = sum(r["ppl"] for r in text_results) / len(text_results)

        results[label] = {
            "avg_ppl": avg_ppl,
            "texts": text_results
        }

    return results


In [ ]:
def execute_ppl_experiment_casualLM(model_name):

    print(f"Executing {model_name}")

    tokenizer, model = load_model_and_tokenizer(models[model_name], AutoModelForCausalLM)

    corpora_files = glob(f"{CORPORA_FOLDER}*.json")
    corpora_files.sort()

    all_results = dict()

    for file_path in corpora_files:
        corpora_name = Path(file_path).stem

        print(f"Processing corpora: {corpora_name}")

        with open(file_path, "r", encoding="utf-8") as f:
            dataset = json.load(f)

        results = eval_causal_sliding(model, tokenizer, dataset)

        print(f"\n{'Language':>10} | {'PPL':>10}")
        print(f"-"*25)
        for language in results.keys():
            print(f"{language:>10} | {results[language]["avg_ppl"] :> 10.3f}")

        all_results[corpora_name] = results

        print("\n", "="*50, "\n")

    results_file = f"{OUTPUT_FOLDER}{model_name}.json"
    print(f"\nSaving results to: {results_file}")

    with open(results_file, 'w') as f:
        json.dump(all_results, f, indent=4, ensure_ascii=False)


    del model
    del tokenizer
    limpar_memoria()

    return all_results


In [ ]:
gemma_results = execute_ppl_experiment_casualLM("Gemma")

In [ ]:
qwen_results = execute_ppl_experiment_casualLM("Qwen")

In [ ]:
tucano_results = execute_ppl_experiment_casualLM("Tucano")

# Masked LM - Pseudo Perplexity


In [4]:
def eval_masked_sliding(model, tokenizer, dataset, stride=510):
    results = {}

    # The pseudo-perplexity is calculated for each text
    for label, texts in dataset.items():
        text_results = []

        for i, text in enumerate(texts):

            inputs = tokenizer(
                text,
                return_tensors="pt",
                add_special_tokens=False
            )

            input_ids_all = inputs["input_ids"][0]
            num_tokens_text = len(input_ids_all)

            total_nll = 0.0
            total_tokens = 0

            # Sliding window
            for begin_window in range(0, num_tokens_text, stride):
                end_window = min(begin_window + stride, num_tokens_text)

                # Select only the tokens from the window
                chunk_ids = input_ids_all[begin_window:end_window]

                # Add special tokens manually: [CLS] text [SEP]
                input_ids = torch.cat([
                    torch.tensor([tokenizer.cls_token_id]),
                    chunk_ids,
                    torch.tensor([tokenizer.sep_token_id])
                ]).unsqueeze(0).to(model.device)

                num_tokens_window = input_ids.size(1)

                # Each token is masked individually
                for token_position in range(1, num_tokens_window - 1):
                    masked_input = input_ids.clone()
                    masked_input[0, token_position] = tokenizer.mask_token_id

                    target_id = input_ids[0, token_position].item()

                    with torch.no_grad():
                        outputs = model(masked_input)
                        logits = outputs.logits

                    log_probs = torch.log_softmax(
                        logits[0, token_position],
                        dim=0
                    )

                    neg_log_likelihood = -log_probs[target_id].item()

                    total_nll += neg_log_likelihood
                    total_tokens += 1

                if end_window == num_tokens_text:
                    break

            pseudo_ppl = math.exp(total_nll / total_tokens)

            text_results.append({
                "text_id": i,
                "num_tokens": num_tokens_text,
                "pseudo_ppl": pseudo_ppl
            })

        avg_pseudo_ppl = sum(r["pseudo_ppl"] for r in text_results) / len(text_results)

        results[label] = {
            "avg_pseudo_ppl": avg_pseudo_ppl,
            "texts": text_results
        }

    return results


In [8]:
def eval_masked_sliding_fast( model, tokenizer, dataset, stride=510, mask_batch_size=16):
    model.eval()
    device = model.device
    results = {}

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    mask_id = tokenizer.mask_token_id

    for label, texts in dataset.items():
        text_results = []

        for text_id, text in enumerate(texts):
            inputs = tokenizer(
                text,
                return_tensors="pt",
                add_special_tokens=False
            )

            input_ids_all = inputs["input_ids"][0].to(device)
            num_tokens_text = input_ids_all.size(0)

            total_nll = 0.0
            total_tokens = 0

            for begin_window in range(0, num_tokens_text, stride):
                end_window = min(begin_window + stride, num_tokens_text)

                chunk_ids = input_ids_all[begin_window:end_window]

                input_ids = torch.cat([
                    torch.tensor([cls_id], device=device),
                    chunk_ids,
                    torch.tensor([sep_id], device=device)
                ]).unsqueeze(0)

                num_tokens_window = input_ids.size(1)

                # Positions excluding [CLS] and [SEP]
                positions = torch.arange(
                    1,
                    num_tokens_window - 1,
                    device=device
                )

                # Process masked positions in batches
                for start in range(0, len(positions), mask_batch_size):
                    batch_positions = positions[start:start + mask_batch_size]
                    batch_size = batch_positions.size(0)

                    # Repeat the same input for each masked position
                    masked_inputs = input_ids.repeat(batch_size, 1)

                    # Mask one different token per row
                    masked_inputs[
                        torch.arange(batch_size, device=device),
                        batch_positions
                    ] = mask_id

                    target_ids = input_ids[0, batch_positions]

                    with torch.no_grad():
                        outputs = model(masked_inputs)
                        logits = outputs.logits

                    # Select logits only at the masked positions
                    masked_logits = logits[
                        torch.arange(batch_size, device=device),
                        batch_positions
                    ]

                    loss = F.cross_entropy(
                        masked_logits,
                        target_ids,
                        reduction="sum"
                    )

                    total_nll += loss.item()
                    total_tokens += batch_size

                if end_window == num_tokens_text:
                    break

            avg_nll = total_nll / total_tokens
            pseudo_ppl = math.exp(avg_nll)

            text_results.append({
                "text_id": text_id,
                "num_tokens": num_tokens_text,
                "pseudo_ppl": pseudo_ppl
            })

        avg_pseudo_ppl = (
            sum(r["pseudo_ppl"] for r in text_results) / len(text_results)
        )

        results[label] = {
            "avg_pseudo_ppl": avg_pseudo_ppl,
            "texts": text_results
        }

    return results

In [9]:
def execute_pseudo_ppl_experiment_maskedLM(model_name):

    print(f"Executing {model_name}")

    tokenizer, model = load_model_and_tokenizer(models[model_name], AutoModelForMaskedLM)

    corpora_files = glob(f"{CORPORA_FOLDER}*.json")
    corpora_files.sort()

    all_results = dict()
    try:
        for file_path in corpora_files:
            corpora_name = Path(file_path).stem

            print(f"Processing corpora: {corpora_name}")

            with open(file_path, "r", encoding="utf-8") as f:
                dataset = json.load(f)

            results = eval_masked_sliding_fast(model, tokenizer, dataset)

            print(f"\n{'Language':>10} | {'PPL':>10}")
            print(f"-"*25)
            for language in results.keys():
                print(f"{language:>10} | {results[language]["avg_pseudo_ppl"] :> 10.3f}")

            all_results[corpora_name] = results

            print("\n", "="*50, "\n")

        results_file = f"{OUTPUT_FOLDER}{model_name}.json"
        print(f"\nSaving results to: {results_file}")

        with open(results_file, 'w') as f:
            json.dump(all_results, f, indent=4, ensure_ascii=False)


    except Exception as e:
        print(e)

    finally:
        del model
        del tokenizer
        limpar_memoria()

    return all_results


In [10]:
execute_pseudo_ppl_experiment_maskedLM("BERTimbau")

Executing BERTimbau


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 16522.62it/s]
[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing corpora: CAPACITAR
🧹 Memória limpa!


KeyboardInterrupt: 